# AI Lab: Two-Group $\mathcal{R}_0$ via the Next-Generation Matrix

**Book companion exercise**

**Considerations exercised:** 7 (mixing structure), 12 (basic vs effective $\mathcal{R}$)
**Estimated runtime:** ~25 minutes
**Companion audit:** [`12_R0_averaging_fallacy.md`](../ai-audit/12_R0_averaging_fallacy.md)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/exercises/ai-lab/12_two_group_NGM_lab.ipynb)


## What this lab does

Constructs the next-generation matrix (NGM) for a two-group model with proportional mixing, computes $\mathcal{R}_0$ as its dominant eigenvalue, and compares against the (incorrect) homogeneous-mean approximation. The numerical contrast confirms why activity-stratified modeling is required for any heterogeneous population.

## Setup

In [ ]:
import sys, os
# AI Lab notebooks live in exercises/ai-lab/ and need access to shared/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'python')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import math
from shared import book_style, BOOK_COLORS
from shared.parameters import baseline_chapter_07
book_style()
rng = np.random.default_rng(42)


## Step 1 — Two-group population

Two activity classes: high ($c_H = 5$/yr, fraction 20%) and low ($c_L = 0.5$/yr, fraction 80%). Per-partnership transmission probability $\beta = 0.05$, infectious period $\tau_R = 10$ years.

In [ ]:
c_H, c_L = 5.0, 0.5
N_H, N_L = 0.20, 0.80
beta = 0.05
tau_R = 10.0

# (1) Homogeneous-mean approximation -- the WRONG answer
c_bar = N_H * c_H + N_L * c_L
R0_hom = c_bar * beta * tau_R
print(f'(1) Homogeneous-mean R_0  = {R0_hom:.3f}  (uses only 1st moment of c)')

# (2) NGM with proportional mixing -- the CORRECT answer
# Probability that a high-activity individual's contact is from each group:
# p(M_HX) = c_X N_X / (c_H N_H + c_L N_L), and similarly for low.
denom = c_H * N_H + c_L * N_L
M = np.array([[c_H * N_H, c_L * N_L],
              [c_H * N_H, c_L * N_L]]) / denom
# Next-generation matrix entry K_ij = beta * c_i * M_ij * tau_R
K = np.array([[beta * c_H * M[0,0] * tau_R, beta * c_H * M[0,1] * tau_R],
              [beta * c_L * M[1,0] * tau_R, beta * c_L * M[1,1] * tau_R]])
eigs = np.linalg.eigvals(K)
R0_ngm = float(np.max(np.real(eigs)))
print(f'(2) NGM dominant eigenvalue R_0 = {R0_ngm:.3f}  (uses 2nd moment of c)')

# Closed-form check: R_0 = (c_H^2 N_H + c_L^2 N_L) / (c_H N_H + c_L N_L) * beta * tau_R
R0_closed = (c_H**2 * N_H + c_L**2 * N_L) / (c_H * N_H + c_L * N_L) * beta * tau_R
print(f'(2 closed-form) R_0       = {R0_closed:.3f}  (must match the NGM eigenvalue)')

print(f'\nRatio R_0_NGM / R_0_hom = {R0_ngm / R0_hom:.2f}')
print(f'(The NGM answer exceeds the homogeneous-mean answer because of activity heterogeneity.)')


## Step 2 — Threshold consequences

In [ ]:
print(f'(1) Homogeneous-mean predicts: R_0 = {R0_hom:.2f} {"BELOW" if R0_hom < 1 else "ABOVE"} threshold (no invasion / invasion)')
print(f'(2) NGM correctly predicts:    R_0 = {R0_ngm:.2f} {"BELOW" if R0_ngm < 1 else "ABOVE"} threshold (no invasion / invasion)')
print()
if R0_hom < 1 and R0_ngm > 1:
    print('** OPERATIONAL CONSEQUENCE: the homogeneous approximation predicts')
    print('   no invasion when in fact the pathogen invades. This is the')
    print('   core-group failure mode visible in the empirical persistence of HIV.')


## Step 3 — Sweep the high-activity fraction

How sensitive is $R_0^{NGM}$ to the high-activity-group fraction?

In [ ]:
N_H_grid = np.linspace(0.01, 0.5, 50)
R0_ngm_sweep = np.zeros_like(N_H_grid)
R0_hom_sweep = np.zeros_like(N_H_grid)
for i, nH in enumerate(N_H_grid):
    nL = 1 - nH
    R0_hom_sweep[i] = (nH * c_H + nL * c_L) * beta * tau_R
    R0_ngm_sweep[i] = (c_H**2 * nH + c_L**2 * nL) / (c_H * nH + c_L * nL) * beta * tau_R

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(N_H_grid * 100, R0_ngm_sweep, color=BOOK_COLORS['primary'], lw=2.4,
        label='NGM (correct)')
ax.plot(N_H_grid * 100, R0_hom_sweep, color=BOOK_COLORS['secondary'], lw=2.4, ls='--',
        label='Homogeneous-mean (incorrect)')
ax.axhline(1.0, color='red', lw=1.2, ls=':', label=r'invasion threshold')
ax.set_xlabel('High-activity fraction $N_H$ (%)')
ax.set_ylabel(r'$\mathcal{R}_0$')
ax.set_title('NGM vs homogeneous-mean as $N_H$ varies (with $c_H = 5/$yr fixed)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Verification: NGM should always exceed homogeneous-mean (Cauchy-Schwarz)
assert np.all(R0_ngm_sweep >= R0_hom_sweep - 1e-10), 'NGM must >= homogeneous-mean'
print('\nVerified: NGM always >= homogeneous-mean (variance contribution is non-negative).')


## What's next

- This NGM construction generalizes to arbitrary group counts; book Ch 9 §9.4 covers the general case
- The HIV case study (Ch 17) applies this exact framework to the activity-stratified bipartite model
- Companion audit: `../ai-audit/12_R0_averaging_fallacy.md`